In [ ]:
# !pip install dwave-ocean-sdk

In [ ]:
# ============================================================
# 1) Imports
# ============================================================
import dimod
import pandas as pd
from dimod import BinaryQuadraticModel, ExactSolver
from dwave.samplers import SimulatedAnnealingSampler


In [ ]:
edge_weights = {
    ('v0', 'v1'): [(0, 4), (60, 4)],
    ('v0', 'v2'): [(0, 8), (60, 8)],
    ('v1', 'v2'): [(0, 8), (20, 8), (35, 20), (60, 20)],
    ('v1', 'v6'): [(0, 5), (20, 5), (30, 18), (60, 5)],
    ('v2', 'v3'): [(0, 15), (60, 15)],
    ('v2', 'v5'): [(0, 5), (60, 5)],
    ('v3', 'v4'): [(0, 6), (60, 6)],
    ('v3', 'v5'): [(0, 22), (20, 22), (35, 6), (60, 6)],
    ('v5', 'v6'): [(0, 8), (25, 8), (45, 12), (60, 12)],
    ('v6', 'v7'): [(0, 2), (60, 2)],
    ('v7', 'v8'): [(0, 3), (60, 3)],
}

adj = {}
for (u, v) in edge_weights:
    adj.setdefault(u, []).append(v)

In [ ]:
"""
Return the weight of edge=(u,v) when departing at time t.
Assumes breakpoints are sorted increasingly.
"""
def w_of_t(edge, t):
    pw = edge_weights[edge]
    current = pw[0][1]
    for bp, wt in pw:
        if t >= bp:
            current = wt
        else:
            break
    return current

In [ ]:
"""
Build the reachable time-expanded graph from (source, t0).
Returns:
  states: set of reachable states (node, time)
  transitions: list of directed transitions
                [((u,t), (v,t2), cost), ...]
"""
def build_time_expanded_graph(source, dest, t0, horizon=100):

    states = set()
    transitions = []

    # simple DFS/BFS over reachable states
    frontier = [(source, t0)]
    states.add((source, t0))

    while frontier:
        u, t = frontier.pop(0)

        if u == dest:
            continue

        for v in adj.get(u, []):
            cost = w_of_t((u, v), t)
            t2 = t + cost

            if t2 <= horizon:
                s1 = (u, t)
                s2 = (v, t2)
                transitions.append((s1, s2, cost))
                if s2 not in states:
                    states.add(s2)
                    frontier.append(s2)

    return states, transitions


In [ ]:
"""
coeffs: dict {var_index: coefficient}
adds penalty * (sum coeffs[i]*x_i - rhs)^2 to Q and offset
"""
def add_squared_constraint(Q, offset, coeffs, rhs, penalty):
    vars_list = sorted(coeffs.keys())

    # diagonal / linear terms
    for i in vars_list:
        ai = coeffs[i]
        Q[(i, i)] = Q.get((i, i), 0.0) + penalty * (ai * ai - 2.0 * rhs * ai)

    # quadratic terms
    for p in range(len(vars_list)):
        for q in range(p + 1, len(vars_list)):
            i, j = vars_list[p], vars_list[q]
            ai, aj = coeffs[i], coeffs[j]
            Q[(i, j)] = Q.get((i, j), 0.0) + penalty * (2.0 * ai * aj)

    # constant offset
    offset += penalty * (rhs * rhs)
    return offset

In [ ]:
# ============================================================
# Build QUBO for one fixed departure time t0
#
#    Variables:
#      one binary variable for each transition in the reachable
#      time-expanded graph
#
#    Objective:
#      minimize total travel time
#
#    Constraints:
#      (a) source state has one more outgoing than incoming
#      (b) all destination states together satisfy destination balance
#      (c) every intermediate state has equal incoming/outgoing flow
# ============================================================
def build_qubo_for_t0(source='v0', dest='v8', t0=0, penalty=100.0):
    states, transitions = build_time_expanded_graph(source, dest, t0)

    # variable indexing: exactly like homework style
    # integer index -> transition
    var_to_transition = {}
    transition_to_var = {}

    for idx, tr in enumerate(transitions):
        transition_to_var[tr] = idx
        var_to_transition[idx] = tr

    Q = {}
    offset = 0.0

    # --------------------------------------------------------
    # Objective term:
    # H_cost = sum cost(e) * x_e
    # --------------------------------------------------------
    for idx, tr in var_to_transition.items():
        _, _, cost = tr
        Q[(idx, idx)] = Q.get((idx, idx), 0.0) + float(cost)

    # --------------------------------------------------------
    # Build incoming / outgoing transition lists per state
    # --------------------------------------------------------
    outgoing = {s: [] for s in states}
    incoming = {s: [] for s in states}

    for idx, tr in var_to_transition.items():
        s1, s2, _ = tr
        outgoing[s1].append(idx)
        incoming[s2].append(idx)

    source_state = (source, t0)
    dest_states = [s for s in states if s[0] == dest]

    # --------------------------------------------------------
    # (A) Source constraint:
    #     sum_out - sum_in = 1
    # --------------------------------------------------------
    coeffs = {}
    for idx in outgoing.get(source_state, []):
        coeffs[idx] = coeffs.get(idx, 0.0) + 1.0
    for idx in incoming.get(source_state, []):
        coeffs[idx] = coeffs.get(idx, 0.0) - 1.0

    offset = add_squared_constraint(Q, offset, coeffs, rhs=1.0, penalty=penalty)

    # --------------------------------------------------------
    # (B) Destination constraint:
    #     aggregate over all destination states
    #     sum_out - sum_in = -1
    # --------------------------------------------------------
    coeffs = {}
    for ds in dest_states:
        for idx in outgoing.get(ds, []):
            coeffs[idx] = coeffs.get(idx, 0.0) + 1.0
        for idx in incoming.get(ds, []):
            coeffs[idx] = coeffs.get(idx, 0.0) - 1.0

    offset = add_squared_constraint(Q, offset, coeffs, rhs=-1.0, penalty=penalty)

    # --------------------------------------------------------
    # (C) Flow conservation for intermediate states:
    #     sum_out - sum_in = 0
    # --------------------------------------------------------
    for s in states:
        if s == source_state:
            continue
        if s[0] == dest:
            continue

        coeffs = {}
        for idx in outgoing.get(s, []):
            coeffs[idx] = coeffs.get(idx, 0.0) + 1.0
        for idx in incoming.get(s, []):
            coeffs[idx] = coeffs.get(idx, 0.0) - 1.0

        if coeffs:
            offset = add_squared_constraint(Q, offset, coeffs, rhs=0.0, penalty=penalty)

    return Q, offset, var_to_transition, states, transitions


In [ ]:
"""
Given the best binary assignment, extract transitions with x=1
and follow them from the source state.
"""
def decode_path(sample, var_to_transition, source='v0', t0=0):

    selected = []
    for idx, val in sample.items():
        if val == 1:
            selected.append(var_to_transition[idx])

    # build map from start-state -> transition
    next_map = {}
    for tr in selected:
        s1, s2, cost = tr
        next_map[s1] = (s2, cost)

    # follow path
    path_states = []
    path_vertices = []
    total_cost = 0

    current = (source, t0)
    visited = set()

    while current in next_map and current not in visited:
        visited.add(current)
        path_states.append(current)
        if not path_vertices:
            path_vertices.append(current[0])

        nxt, c = next_map[current]
        total_cost += c
        path_vertices.append(nxt[0])
        current = nxt

    path_states.append(current)

    return selected, path_states, path_vertices, total_cost

In [ ]:
def run_experiment(t0, penalty=100.0, method="exact", num_reads=2000):
    print("=" * 70)
    print(f"Experiment for departure time t0 = {t0}")
    print("=" * 70)

    Q, offset, var_to_transition, states, transitions = build_qubo_for_t0(
        source='v0', dest='v8', t0=t0, penalty=penalty
    )

    print("\nNumber of reachable states:", len(states))
    print("Number of time-expanded transitions / QUBO variables:", len(var_to_transition))

    print("\nVariable indexing:")
    for idx in sorted(var_to_transition):
        s1, s2, c = var_to_transition[idx]
        print(f"  {idx}: {s1} -> {s2}, cost={c}")

    # Build BQM directly from QUBO
    bqm = BinaryQuadraticModel.from_qubo(Q, offset=offset)

    print("\nSolving method:", method)

    # --------------------------------------------------------
    # Solve
    # --------------------------------------------------------
    if method == "exact":
        sampler = ExactSolver()
        sampleset = sampler.sample(bqm)
    elif method == "sa":
        sampler = SimulatedAnnealingSampler()
        sampleset = sampler.sample(bqm, num_reads=num_reads)
    else:
        raise ValueError("method must be 'exact' or 'sa'")

    # --------------------------------------------------------
    # Only inspect the best result
    # --------------------------------------------------------
    best_x = sampleset.first.sample
    best_energy = sampleset.first.energy

    # --------------------------------------------------------
    # Show only the lowest-energy solutions
    # --------------------------------------------------------
    print("\nLowest-energy solutions (top 100):")

    # Convert just the needed part to a dataframe-like view
    top_k = 100
    shown = 0

    print("sample_index | energy | num_occurrences | selected variables")
    print("-" * 80)

    for idx, row in enumerate(sampleset.data(['sample', 'energy', 'num_occurrences'])):
        if shown >= top_k:
            break

        sample = row.sample
        energy = row.energy
        num_occ = row.num_occurrences

        selected_vars = [i for i, v in sample.items() if v == 1]

        print(f"{idx:11d} | {energy:6.1f} | {num_occ:15d} | {selected_vars}")
        shown += 1

    # --------------------------------------------------------
    # Decode path
    # --------------------------------------------------------
    selected, path_states, path_vertices, total_cost = decode_path(
        best_x, var_to_transition, source='v0', t0=t0
    )

    print("\nDecoded time-expanded path states:")
    for s in path_states:
        print(" ", s)

    print("\nDecoded path in original graph:")
    print(" -> ".join(path_vertices))

    print("Decoded total travel cost:", total_cost)

    return {
        "sampleset": sampleset,
        "best_x": best_x,
        "best_energy": best_energy,
        "selected_vars": selected_vars,
        "selected_transitions": selected,
        "path_states": path_states,
        "path_vertices": path_vertices,
        "total_cost": total_cost,
    }


In [ ]:
# Run the three scenarios

# 1) Exact validation on the first scenario
res_0 = run_experiment(t0=0, penalty=100.0, method="exact")
res_30 = run_experiment(t0=30, penalty=100.0, method="exact")
res_50 = run_experiment(t0=50, penalty=100.0, method="exact")


# 2) Faster approximate runs for the remaining scenarios
#res_30 = run_experiment(t0=30, penalty=100.0, method="sa", num_reads=5000)
#res_50 = run_experiment(t0=50, penalty=100.0, method="sa", num_reads=5000)


#res_0  = run_experiment(t0=0,  penalty=100.0, method="sa", num_reads=5000)
#res_30 = run_experiment(t0=30, penalty=100.0, method="sa", num_reads=5000)
#res_50 = run_experiment(t0=50, penalty=100.0, method="sa", num_reads=5000)

Experiment for departure time t0 = 0

Number of reachable states: 27
Number of time-expanded transitions / QUBO variables: 26

Variable indexing:
  0: ('v0', 0) -> ('v1', 4), cost=4
  1: ('v0', 0) -> ('v2', 8), cost=8
  2: ('v1', 4) -> ('v2', 12), cost=8
  3: ('v1', 4) -> ('v6', 9), cost=5
  4: ('v2', 8) -> ('v3', 23), cost=15
  5: ('v2', 8) -> ('v5', 13), cost=5
  6: ('v2', 12) -> ('v3', 27), cost=15
  7: ('v2', 12) -> ('v5', 17), cost=5
  8: ('v6', 9) -> ('v7', 11), cost=2
  9: ('v3', 23) -> ('v4', 29), cost=6
  10: ('v3', 23) -> ('v5', 45), cost=22
  11: ('v5', 13) -> ('v6', 21), cost=8
  12: ('v3', 27) -> ('v4', 33), cost=6
  13: ('v3', 27) -> ('v5', 49), cost=22
  14: ('v5', 17) -> ('v6', 25), cost=8
  15: ('v7', 11) -> ('v8', 14), cost=3
  16: ('v5', 45) -> ('v6', 57), cost=12
  17: ('v6', 21) -> ('v7', 23), cost=2
  18: ('v5', 49) -> ('v6', 61), cost=12
  19: ('v6', 25) -> ('v7', 27), cost=2
  20: ('v6', 57) -> ('v7', 59), cost=2
  21: ('v7', 23) -> ('v8', 26), cost=3
  22: ('v6